<div style="background: linear-gradient(135deg, #0a0a0a 0%, #1a1a2e 30%, #16213e 60%, #0f3460 100%); padding: 55px 40px; border-radius: 18px; text-align: center; margin-bottom: 30px; border: 1px solid #e94560;">
  <h1 style="color: #e94560; font-size: 2.6em; font-weight: 900; letter-spacing: 3px; margin: 0;">🌍 SEABORN MASTERCLASS</h1>
  <h2 style="color: #f5f5f5; font-size: 1.7em; font-weight: 400; margin-top: 12px;">Notebook 6 — Real-World Case Studies</h2>
  <p style="color: #a8dadc; font-size: 1.05em; margin-top: 16px; letter-spacing: 1px;">MASTER EDA NOTEBOOK — Every Concept. Every Plot. Real Business Context.</p>
  <hr style="border: 1px solid #e94560; margin: 22px auto; width: 55%;">
  <p style="color: #888; font-size: 0.95em;">📦 Notebooks 1–5 Concepts Unified &nbsp;|&nbsp; ⏱ Est. Time: 180 mins &nbsp;|&nbsp; 🎯 Interview-Ready Portfolio</p>
</div>

---
## 🎓 About This Notebook

> This is the **Master Case Study Notebook**. Unlike Notebooks 1–5 which taught concepts **by plot type**, this notebook applies **every concept together** in realistic business EDA workflows — the way you'd actually use Seaborn on the job or in an interview project.

### 🗺️ Concepts Revisited Here (from all previous notebooks)
| From NB | Concepts Applied |
|---------|----------------|
| NB1 | `set_theme`, styles, palettes, color theory |
| NB2 | `histplot`, `kdeplot`, `countplot`, `barplot`, `boxplot`, `violinplot` |
| NB3 | `scatterplot`, `lineplot`, `relplot`, `regplot`, `lmplot`, CI anatomy |
| NB4 | `heatmap` (masked), missing value audit, `pairplot`, `jointplot`, `clustermap`, feature ranking |
| NB5 | `FacetGrid`, `PairGrid`, `twinx`, annotations, reusable functions |

### 📁 Four Case Studies
| # | Dataset | Business Domain | Key Question |
|---|---------|----------------|-------------|
| 1 | `tips` | 🍽️ Restaurant Analytics | What drives tip amount? How to optimize revenue? |
| 2 | `titanic` | 🚢 Survival Analysis | Which passenger features predict survival? |
| 3 | `diamonds` | 💎 Luxury Retail Pricing | What drives diamond price? Pricing strategy? |
| 4 | `flights` | ✈️ Aviation Growth | Seasonal patterns and growth trends? |

### 🚦 Difficulty Legend
| Marker | Level |
|--------|-------|
| 🟢 Basic | Core syntax — must know |
| 🟡 Intermediate | Parameters and combinations |
| 🔴 Advanced | Production patterns |

---

## ⚙️ Global Setup — Run This First

In [ ]:
# 🟢 Basic | One-time global setup — applies to all 4 case studies
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import os

# ── Professional global theme (NB1 concept) ──
sns.set_theme(
    style='whitegrid',
    context='notebook',
    font_scale=1.08,
    rc={
        'figure.facecolor':  'white',
        'axes.facecolor':    '#FAFAFA',
        'grid.alpha':         0.35,
        'figure.dpi':         110,
        'axes.spines.top':    False,
        'axes.spines.right':  False,
        'axes.titlepad':      12,
    }
)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
pd.set_option('display.max_columns', None)
os.makedirs('outputs/case_studies', exist_ok=True)

# ── Reusable EDA helper (NB5 concept) ──
def quick_data_summary(df: pd.DataFrame, name: str) -> None:
    """Print a fast EDA summary — shape, dtypes, missing values."""
    print(f'\n{"="*55}')
    print(f'  DATASET: {name}')
    print(f'{"="*55}')
    print(f'  Shape    : {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'  Columns  : {list(df.columns)}')
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing):
        print(f'  Missing  :')
        pct = (missing / len(df) * 100).round(1)
        print(pd.DataFrame({"count": missing, "pct%": pct}).to_string())
    else:
        print('  Missing  : None ✅')
    num_cols = df.select_dtypes(include="number").columns.tolist()
    print(f'  Numeric  : {num_cols}')
    print(f'{"="*55}')

print('✅ Global setup complete. Ready for Case Studies.')

---

<div style="background: linear-gradient(90deg, #1b5e20, #2e7d32); padding: 20px 30px; border-radius: 12px; margin: 20px 0;">
  <h2 style="color: white; margin: 0; font-size: 1.8em;">🍽️ Case Study 1 — Restaurant Analytics</h2>
  <p style="color: #c8e6c9; margin: 8px 0 0 0; font-size: 1.05em;">Dataset: <code>tips</code> | Goal: Understand tipping behavior and identify revenue optimization opportunities</p>
</div>

### 🏢 Business Context
A restaurant chain wants to understand:
1. What is the distribution of bills and tips?
2. Which factors most influence tip amount?
3. Are there patterns by day, time, or party size?
4. Can we predict tip amount from bill size?

### 📋 EDA Checklist
- [ ] Data overview (shape, missing, dtypes)
- [ ] Univariate distributions
- [ ] Categorical breakdowns
- [ ] Bivariate relationships
- [ ] Multivariate patterns
- [ ] Regression analysis
- [ ] Business recommendations

In [ ]:
# 🟢 Basic | Step 1: Data Overview (NB4 — missing value audit)
tips = sns.load_dataset('tips')
tips['tip_pct']  = (tips['tip']  / tips['total_bill'] * 100).round(2)
tips['bill_per_person'] = (tips['total_bill'] / tips['size']).round(2)

quick_data_summary(tips, 'Restaurant Tips')
print()
print(tips.describe().round(2).to_string())

In [ ]:
# 🟡 Intermediate | Step 2: Univariate Analysis (NB2 — histplot, kdeplot)
# How are bills and tips distributed? Are they skewed?

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ── Panel 1: Bill distribution + stats ──
sns.histplot(data=tips, x='total_bill', kde=True, bins=22,
             color='#1565C0', alpha=0.65, ax=axes[0, 0])
axes[0, 0].axvline(tips['total_bill'].mean(),   color='#E53935', lw=2, linestyle='--',
                   label=f'Mean: ${tips["total_bill"].mean():.2f}')
axes[0, 0].axvline(tips['total_bill'].median(), color='#2E7D32', lw=2, linestyle='-.',
                   label=f'Median: ${tips["total_bill"].median():.2f}')
axes[0, 0].set_title('① Bill Distribution
(Right-skewed: mean > median)', fontweight='bold')
axes[0, 0].set_xlabel('Total Bill ($)'); axes[0, 0].legend(fontsize=9)

# ── Panel 2: Tip percentage KDE by meal time ──
sns.kdeplot(data=tips, x='tip_pct', hue='time',
            fill=True, alpha=0.45, common_norm=False,
            palette={'Lunch': '#1565C0', 'Dinner': '#E53935'}, ax=axes[0, 1])
axes[0, 1].axvline(15, color='#FB8C00', lw=2, linestyle='--', label='15% standard')
axes[0, 1].axvline(20, color='#2E7D32', lw=2, linestyle='--', label='20% generous')
axes[0, 1].set_title('② Tip % by Meal Time
(Dinner vs Lunch distribution)', fontweight='bold')
axes[0, 1].set_xlabel('Tip Percentage (%)'); axes[0, 1].legend(fontsize=8)

# ── Panel 3: Count by day (sorted) ──
day_order = tips['day'].value_counts().index.tolist()
sns.countplot(data=tips, x='day', order=day_order, hue='day',
              palette='Set2', legend=False, ax=axes[0, 2])
day_counts = tips['day'].value_counts().reindex(day_order)
total_tips  = len(tips)
axes[0, 2].bar_label(
    axes[0, 2].containers[0],
    labels=[
        f'{day_counts.iloc[0]}
({day_counts.iloc[0]/total_tips*100:.1f}%)',
        f'{day_counts.iloc[1]}
({day_counts.iloc[1]/total_tips*100:.1f}%)',
        f'{day_counts.iloc[2]}
({day_counts.iloc[2]/total_tips*100:.1f}%)',
        f'{day_counts.iloc[3]}
({day_counts.iloc[3]/total_tips*100:.1f}%)',
    ],
    fontsize=9, fontweight='bold', padding=3
)
axes[0, 2].set_title('③ Visits by Day
(Saturday = busiest)', fontweight='bold')
axes[0, 2].set_xlabel('Day'); axes[0, 2].set_ylabel('Count')

# ── Panel 4: Boxplot — bill by day (NB2) ──
sns.boxplot(data=tips, x='day', y='total_bill', order=day_order,
            hue='day', palette='Set2', legend=False,
            medianprops=dict(color='black', linewidth=2.5),
            width=0.55, ax=axes[1, 0])
axes[1, 0].set_title('④ Bill Distribution by Day
(IQR + outliers visible)', fontweight='bold')
axes[1, 0].set_xlabel('Day'); axes[1, 0].set_ylabel('Total Bill ($)')

# ── Panel 5: Violin — tip by smoker (NB2) ──
sns.violinplot(data=tips, x='smoker', y='tip', hue='smoker',
               palette={'Yes': '#E53935', 'No': '#1565C0'},
               inner='box', split=False, legend=False, ax=axes[1, 1])
axes[1, 1].set_title('⑤ Tip by Smoker Status
(Shape = distribution | Box = IQR)', fontweight='bold')
axes[1, 1].set_xlabel('Smoker?'); axes[1, 1].set_ylabel('Tip ($)')

# ── Panel 6: Bill per person by party size ──
sns.barplot(data=tips, x='size', y='bill_per_person',
            hue='size', palette='Blues_d', estimator='mean',
            errorbar=('ci', 95), legend=False, ax=axes[1, 2])
axes[1, 2].set_title('⑥ Avg Bill/Person by Party Size
(Larger parties = cheaper per head)', fontweight='bold')
axes[1, 2].set_xlabel('Party Size'); axes[1, 2].set_ylabel('Bill per Person ($)')

fig.suptitle('🍽️ Case Study 1 — Restaurant Univariate & Categorical Analysis',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig('outputs/case_studies/cs1_univariate.png', dpi=140, bbox_inches='tight')
plt.show()

### 📋 Output Preview — Step 2
```
Key findings from 6 panels:
① Bills right-skewed: Mean $19.79 > Median $17.80 → outlier large tables
② Tip %: Both Lunch and Dinner peak near 15–18% — similar behavior
③ Saturday = 35.7% of all visits, Friday = fewest (7.8%)
④ Saturday has widest IQR — highest variance in bill sizes
⑤ Smoker/non-smoker tip distributions nearly identical
⑥ Bill per person DROPS as party grows — economies of scale in ordering
```

In [ ]:
# 🟡 Intermediate | Step 3: Bivariate & Correlation Analysis (NB3, NB4)
# Which features are correlated? What predicts tip amount?

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── Panel 1: Scatter — bill vs tip, annotated (NB3 + NB5) ──
sns.scatterplot(data=tips, x='total_bill', y='tip',
                hue='time', style='smoker',
                palette={'Lunch': '#1565C0', 'Dinner': '#E53935'},
                alpha=0.65, s=55, ax=axes[0])

max_tip_row  = tips.loc[tips['tip'].idxmax()]
max_bill_row = tips.loc[tips['total_bill'].idxmax()]

axes[0].annotate(
    f"Best tipper
${max_tip_row['tip']:.2f} tip",
    xy=(max_tip_row['total_bill'], max_tip_row['tip']),
    xytext=(max_tip_row['total_bill'] - 11, max_tip_row['tip'] + 0.2),
    arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.8),
    fontsize=8.5, color='#2E7D32', fontweight='bold',
    bbox=dict(boxstyle='round', fc='#E8F5E9', alpha=0.9, ec='#2E7D32')
)
axes[0].annotate(
    f"Biggest bill
${max_bill_row['total_bill']:.2f}",
    xy=(max_bill_row['total_bill'], max_bill_row['tip']),
    xytext=(max_bill_row['total_bill'] - 16, max_bill_row['tip'] + 1.5),
    arrowprops=dict(arrowstyle='->', color='#E53935', lw=1.8,
                    connectionstyle='arc3,rad=-0.2'),
    fontsize=8.5, color='#C62828', fontweight='bold',
    bbox=dict(boxstyle='round', fc='#FFEBEE', alpha=0.9, ec='#E53935')
)
axes[0].set_title('① Bill vs Tip (annotated outliers)', fontweight='bold')
axes[0].set_xlabel('Total Bill ($)'); axes[0].set_ylabel('Tip ($)')
axes[0].legend(title='Time/Smoker', fontsize=8, loc='upper left')

# ── Panel 2: Masked correlation heatmap (NB4) ──
tips_num  = tips[['total_bill', 'tip', 'size', 'tip_pct', 'bill_per_person']]
corr_tips = tips_num.corr()
mask_tips = np.triu(np.ones_like(corr_tips, dtype=bool))
sns.heatmap(corr_tips, mask=mask_tips, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.8,
            cbar_kws={'shrink': 0.75}, ax=axes[1])
axes[1].set_title('② Feature Correlation Matrix
(masked upper triangle)', fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right', fontsize=9)

# ── Panel 3: Feature ranking — correlation with tip (NB4) ──
corr_with_tip = (corr_tips['tip'].drop('tip').abs()
                                 .sort_values(ascending=True))
bar_cols = pd.Series(corr_with_tip.values).map(
    lambda v: '#E53935' if v > 0.5 else '#FB8C00' if v > 0.3 else '#FDD835'
).tolist()
b3 = axes[2].barh(corr_with_tip.index, corr_with_tip.values,
                   color=bar_cols, edgecolor='white')
axes[2].bar_label(b3, fmt='%.3f', fontsize=10, fontweight='bold', padding=4)
axes[2].set_title('③ Feature Correlation with Tip
(top feature = total_bill)', fontweight='bold')
axes[2].set_xlabel('|Pearson r| with tip')
axes[2].axvline(0.5, color='#E53935', lw=1.3, linestyle='--', alpha=0.7)
axes[2].axvline(0.3, color='#FB8C00', lw=1.3, linestyle='--', alpha=0.7)

fig.suptitle('🍽️ Case Study 1 — Bivariate & Correlation Analysis',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig('outputs/case_studies/cs1_bivariate.png', dpi=140, bbox_inches='tight')
plt.show()

print(f"📊 CORRELATION FINDINGS:")
print(corr_tips['tip'].drop('tip').sort_values(key=abs, ascending=False).to_string())
print()
print("⚠️  WARNING: tip_pct is derived FROM tip — data leakage! Don't use as predictor.")

In [ ]:
# 🔴 Advanced | Step 4: Regression + FacetGrid (NB3 + NB5)
# Regression for prediction. FacetGrid to check consistency across subgroups.

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: regplot — linear vs LOWESS (NB3) ──
sns.regplot(data=tips, x='total_bill', y='tip', ci=95,
            scatter_kws={'alpha': 0.40, 's': 35, 'color': '#888'},
            line_kws={'color': '#1565C0', 'lw': 2.5, 'label': 'Linear fit'},
            ax=axes[0])
sns.regplot(data=tips, x='total_bill', y='tip', ci=None, scatter=False,
            lowess=True,
            line_kws={'color': '#E53935', 'lw': 2.5, 'linestyle': '--',
                      'label': 'LOWESS (actual shape)'},
            ax=axes[0])

corr_val = tips['total_bill'].corr(tips['tip'])
r_sq = corr_val ** 2
axes[0].annotate(
    f'R² = {r_sq:.3f}
Model explains
{r_sq*100:.1f}% of tip variance',
    xy=(0.05, 0.88), xycoords='axes fraction',
    fontsize=9.5, fontweight='bold', color='#1565C0',
    bbox=dict(boxstyle='round', fc='#E3F2FD', alpha=0.9, ec='#1565C0')
)
axes[0].set_title(f'④ Regression: Bill → Tip
(Linear ≈ LOWESS → linear model appropriate)',
                  fontweight='bold')
axes[0].set_xlabel('Total Bill ($)'); axes[0].set_ylabel('Tip ($)')
axes[0].legend(fontsize=9)

# ── Right: lmplot-style per day using barplot with CI ──
sns.barplot(data=tips, x='day', y='tip', hue='sex', order=day_order,
            palette={'Male': '#1565C0', 'Female': '#C62828'},
            estimator='mean', errorbar=('ci', 95), ax=axes[1])
axes[1].set_title('⑤ Avg Tip by Day × Gender
(Error bars = 95% CI — overlap = no significant diff)',
                  fontweight='bold')
axes[1].set_xlabel('Day'); axes[1].set_ylabel('Average Tip ($)')
axes[1].legend(title='Gender', fontsize=9)

fig.suptitle('🍽️ Case Study 1 — Regression & Group Comparison',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig('outputs/case_studies/cs1_regression.png', dpi=140, bbox_inches='tight')
plt.show()

print(f"📊 R² = {r_sq:.3f} — total_bill explains {r_sq*100:.1f}% of tip variance.")
print("   Remaining 54% from: service quality, generosity, group dynamics, occasion.")
print("   → Simple linear model: tip ≈ 0.105 × total_bill + 0.92")

In [ ]:
# 🔴 Advanced | Step 5: Revenue Dashboard — twinx + FacetGrid (NB5)
# Business insight: visualise daily patterns to optimize staffing and pricing.

# ── Chart A: Daily volume + avg tip (twinx) ──
daily_stats = tips.groupby('day').agg(
    visits=('tip', 'count'),
    avg_tip=('tip', 'mean'),
    avg_bill=('total_bill', 'mean')
).reindex(['Thur', 'Fri', 'Sat', 'Sun'])

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

bars = ax1.bar(daily_stats.index, daily_stats['visits'],
               color='#90CAF9', alpha=0.75, width=0.5, label='Visit Count')
ax2.plot(daily_stats.index, daily_stats['avg_tip'],
         color='#E53935', linewidth=2.5, marker='D', markersize=9,
         markerfacecolor='white', markeredgecolor='#E53935', markeredgewidth=2,
         label='Avg Tip ($)')

ax1.set_ylabel('Number of Visits', fontsize=11, color='#1565C0')
ax2.set_ylabel('Average Tip ($)',   fontsize=11, color='#E53935')
ax1.tick_params(axis='y', labelcolor='#1565C0')
ax2.tick_params(axis='y', labelcolor='#E53935')
ax1.set_ylim(0, daily_stats['visits'].max() * 1.25)
ax2.set_ylim(0, daily_stats['avg_tip'].max() * 1.4)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=10)
ax1.set_title('🍽️ Restaurant KPI Dashboard — Visits (bars) vs Avg Tip (line)
'
              '(twinx: two metrics, one chart)', fontsize=12, fontweight='bold', pad=12)

# Annotate best day
best_day = daily_stats['avg_tip'].idxmax()
best_val = daily_stats.loc[best_day, 'avg_tip']
ax2.annotate(f'Best avg tip
${best_val:.2f}',
             xy=(best_day, best_val),
             xytext=(best_day, best_val + 0.35),
             fontsize=9, color='#C62828', fontweight='bold',
             bbox=dict(boxstyle='round', fc='#FFEBEE', alpha=0.9, ec='#E53935'))

plt.tight_layout()
fig.savefig('outputs/case_studies/cs1_kpi_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()

print("📊 BUSINESS RECOMMENDATIONS (Restaurant):")
print(f"   1. Deploy maximum staff on Saturday ({daily_stats.loc['Sat','visits']} visits, highest volume)")
print(f"   2. {best_day} has highest avg tip (${best_val:.2f}) — consider weekend pricing strategy")
print(f"   3. Friday is slowest ({daily_stats.loc['Fri','visits']} visits) — promo deals recommended")
print(f"   4. Bill → Tip is linear (R²={r_sq:.2f}) — implement automatic tip suggestions on receipts")

---

<div style="background: linear-gradient(90deg, #0d47a1, #1565c0); padding: 20px 30px; border-radius: 12px; margin: 20px 0;">
  <h2 style="color: white; margin: 0; font-size: 1.8em;">🚢 Case Study 2 — Titanic Survival Analysis</h2>
  <p style="color: #bbdefb; margin: 8px 0 0 0; font-size: 1.05em;">Dataset: <code>titanic</code> | Goal: Understand survival patterns to build an ML classification model</p>
</div>

### 🏢 Business Context
An insurance company wants to understand historical maritime survival patterns:
1. What demographic features predict survival?
2. Is the data complete enough for modelling?
3. Which features show the clearest survival signal?
4. What are the ethical patterns in survival rates?

In [ ]:
# 🟢 Basic | CS2 Step 1: Data Overview + Missing Value Audit (NB4)
titanic = sns.load_dataset('titanic')
quick_data_summary(titanic, 'Titanic Survival Dataset')

In [ ]:
# 🟡 Intermediate | CS2 Step 2: Missing Values + Survival Distributions (NB2, NB4)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ── Panel 1: Missing value heatmap (NB4) ──
key_cols = ['survived', 'pclass', 'sex', 'age', 'fare', 'embarked', 'deck']
sns.heatmap(titanic[key_cols].isnull(), yticklabels=False, cbar=False,
            cmap='gray_r', ax=axes[0, 0])
axes[0, 0].set_title('① Missing Value Map
(Dark = missing)', fontweight='bold')

# ── Panel 2: Survival rate by class (countplot) ──
sns.countplot(data=titanic, x='pclass', hue='survived',
              palette={0: '#E53935', 1: '#2E7D32'}, ax=axes[0, 1])
axes[0, 1].set_title('② Survival Count by Class
(Green=survived | Red=died)', fontweight='bold')
axes[0, 1].set_xlabel('Passenger Class'); axes[0, 1].set_ylabel('Count')
axes[0, 1].legend(title='Survived', labels=['No', 'Yes'], fontsize=9)

# ── Panel 3: Survival rate (%) by class — barplot ──
surv_by_class = titanic.groupby('pclass')['survived'].mean().reset_index()
surv_by_class.columns = ['pclass', 'survival_rate']
surv_by_class['pclass'] = surv_by_class['pclass'].map({1: '1st', 2: '2nd', 3: '3rd'})
bars_sc = axes[0, 2].bar(surv_by_class['pclass'], surv_by_class['survival_rate'] * 100,
                          color=['#2E7D32', '#FB8C00', '#E53935'], width=0.5, edgecolor='white')
axes[0, 2].bar_label(bars_sc,
    labels=[f'{surv_by_class["survival_rate"].iloc[0]*100:.1f}%',
            f'{surv_by_class["survival_rate"].iloc[1]*100:.1f}%',
            f'{surv_by_class["survival_rate"].iloc[2]*100:.1f}%'],
    fontsize=11, fontweight='bold', padding=4)
axes[0, 2].set_title('③ Survival Rate by Class
(1st class 3× more likely to survive)', fontweight='bold')
axes[0, 2].set_ylabel('Survival Rate (%)'); axes[0, 2].set_ylim(0, 80)

# ── Panel 4: Age KDE by survival (NB2 + NB3) ──
sns.kdeplot(data=titanic.dropna(subset=['age']), x='age', hue='survived',
            fill=True, alpha=0.45, common_norm=False,
            palette={0: '#E53935', 1: '#2E7D32'}, ax=axes[1, 0])
axes[1, 0].set_title('④ Age Distribution by Survival
(Young children had higher survival)', fontweight='bold')
axes[1, 0].set_xlabel('Age'); axes[1, 0].legend(title='Survived', labels=['No','Yes'], fontsize=9)

# ── Panel 5: Fare KDE by survival (NB2) ──
sns.kdeplot(data=titanic.dropna(subset=['fare']), x='fare', hue='survived',
            fill=True, alpha=0.45, common_norm=False, clip=(0, 200),
            palette={0: '#E53935', 1: '#2E7D32'}, ax=axes[1, 1])
axes[1, 1].set_title('⑤ Fare Distribution by Survival
(Higher fare → higher survival)', fontweight='bold')
axes[1, 1].set_xlabel('Fare (£)')
axes[1, 1].legend(title='Survived', labels=['No','Yes'], fontsize=9)

# ── Panel 6: Violin — fare by class × survived ──
sns.violinplot(data=titanic.dropna(subset=['fare']), x='pclass', y='fare',
               hue='survived', split=True, inner='box',
               palette={0: '#E53935', 1: '#2E7D32'}, ax=axes[1, 2])
axes[1, 2].set_title('⑥ Fare by Class (split violin)
Survivors paid more within each class', fontweight='bold')
axes[1, 2].set_xlabel('Passenger Class'); axes[1, 2].set_ylabel('Fare (£)')
axes[1, 2].set_ylim(-5, 220)
axes[1, 2].legend(title='Survived', labels=['No','Yes'], fontsize=9)

fig.suptitle('🚢 Case Study 2 — Titanic: Distributions & Survival Patterns',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig('outputs/case_studies/cs2_distributions.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# 🔴 Advanced | CS2 Step 3: Multivariate + Feature Analysis (NB4 + NB5)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── Panel 1: Masked correlation heatmap (NB4) ──
titanic_num = titanic[['survived','pclass','age','sibsp','parch','fare']].dropna()
corr_t      = titanic_num.corr()
mask_t      = np.triu(np.ones_like(corr_t, dtype=bool))
sns.heatmap(corr_t, mask=mask_t, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.8,
            cbar_kws={'shrink': 0.75}, ax=axes[0])
axes[0].set_title('① Correlation Heatmap (masked)
(pclass = strongest survival predictor)',
                  fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right', fontsize=8)

# ── Panel 2: Feature correlation with survival ──
surv_corr = (corr_t['survived'].drop('survived')
                               .sort_values(key=abs, ascending=True))
bar_colors_t = pd.Series(surv_corr.values).map(
    lambda v: '#E53935' if abs(v) > 0.3 else '#FB8C00' if abs(v) > 0.15 else '#FDD835'
).tolist()
b2 = axes[1].barh(surv_corr.index, surv_corr.abs().values,
                   color=bar_colors_t, edgecolor='white')
axes[1].bar_label(b2, fmt='%.3f', fontsize=10, fontweight='bold', padding=4)
axes[1].set_title('② Feature Correlation with Survival
(pclass most predictive)', fontweight='bold')
axes[1].set_xlabel('|Pearson r| with survived')

# ── Panel 3: Survival rate heatmap by class + sex (2D) ──
survival_matrix = titanic.groupby(['pclass', 'sex'])['survived'].mean().unstack()
sns.heatmap(survival_matrix * 100, annot=True, fmt='.1f',
            cmap='RdYlGn', vmin=0, vmax=100,
            linewidths=1, linecolor='white',
            cbar_kws={'label': 'Survival Rate (%)', 'shrink': 0.75},
            ax=axes[2])
axes[2].set_title('③ Survival Rate Matrix: Class × Sex
(Most powerful single chart for Titanic)',
                  fontweight='bold')
axes[2].set_xlabel('Sex'); axes[2].set_ylabel('Passenger Class')

fig.suptitle('🚢 Case Study 2 — Titanic: Multivariate Analysis',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig('outputs/case_studies/cs2_multivariate.png', dpi=140, bbox_inches='tight')
plt.show()

print("📊 SURVIVAL RATE MATRIX INSIGHTS:")
print(survival_matrix.round(3).to_string())
print()
print("🎯 Key finding: 1st class female = ~97% survival | 3rd class male = ~17%")
print("   → Class + Sex are the two most powerful features for a classification model.")
print("   → This is why Logistic Regression on just these 2 features achieves ~80% accuracy.")

In [ ]:
# 🔴 Advanced | CS2 Step 4: FacetGrid — survival breakdown (NB5)
# Show how survival patterns change across ALL combinations of class and sex.

g = sns.FacetGrid(
    titanic.dropna(subset=['age']),
    col='pclass',
    row='sex',
    height=3.5,
    aspect=1.2,
    margin_titles=True
)
g.map_dataframe(sns.histplot, x='age', hue='survived',
                bins=15, alpha=0.55,
                palette={0: '#E53935', 1: '#2E7D32'})
g.add_legend(title='Survived', labels=['No', 'Yes'])
g.set_axis_labels('Age', 'Count')
g.set_titles(col_template='Class {col_name}', row_template='{row_name}')
g.figure.suptitle(
    '🚢 Age Distribution by Survival — Across ALL Class × Sex Combinations
'
    '(FacetGrid: map_dataframe applied to 6 subgroups simultaneously)',
    fontsize=12, fontweight='bold', y=1.03
)
plt.tight_layout()
g.figure.savefig('outputs/case_studies/cs2_facetgrid.png', dpi=140, bbox_inches='tight')
plt.show()

print("📊 PATTERN: Young children (age 0–10) show green bars in all panels — protected.")
print("   1st class females: almost entirely green (survived) across all ages.")
print("   3rd class males: almost entirely red (died) especially 20–40 age range.")

---

<div style="background: linear-gradient(90deg, #4a148c, #6a1b9a); padding: 20px 30px; border-radius: 12px; margin: 20px 0;">
  <h2 style="color: white; margin: 0; font-size: 1.8em;">💎 Case Study 3 — Diamond Price Analysis</h2>
  <p style="color: #e1bee7; margin: 8px 0 0 0; font-size: 1.05em;">Dataset: <code>diamonds</code> (53,940 rows) | Goal: Build pricing strategy from data</p>
</div>

### 🏢 Business Context
A luxury jewellery retailer wants to:
1. Understand what drives diamond price
2. Build a data-driven pricing model
3. Find underpriced and overpriced segments
4. Identify the most profitable cut/color/clarity combinations

In [ ]:
# 🟢 Basic | CS3 Step 1: Overview (NB1, NB4)
diamonds = sns.load_dataset('diamonds')
diamonds['price_per_carat'] = (diamonds['price'] / diamonds['carat']).round(0)

quick_data_summary(diamonds, 'Diamonds Pricing Dataset')
print()
print(diamonds[['carat','price','price_per_carat']].describe().round(1).to_string())

In [ ]:
# 🟡 Intermediate | CS3 Step 2: Price & Feature Distributions (NB2)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ── Panel 1: Price distribution — raw ──
sns.histplot(data=diamonds, x='price', bins=50, kde=True,
             color='#6A1B9A', alpha=0.65, ax=axes[0, 0])
axes[0, 0].axvline(diamonds['price'].median(), color='#E53935', lw=2,
                   linestyle='--', label=f'Median: ${diamonds["price"].median():,.0f}')
axes[0, 0].set_title('① Price Distribution
(Strongly right-skewed)', fontweight='bold')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0, 0].legend(fontsize=9)

# ── Panel 2: Count by cut quality ──
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
cut_counts = diamonds['cut'].value_counts().reindex(cut_order)
sns.countplot(data=diamonds, x='cut', order=cut_order, hue='cut',
              palette='Purples', legend=False, ax=axes[0, 1])
axes[0, 1].set_title('② Diamond Count by Cut Quality
(Ideal is most common)', fontweight='bold')
axes[0, 1].set_xlabel('Cut Quality'); axes[0, 1].set_ylabel('Count')

# ── Panel 3: Avg price by cut — counterintuitive! ──
sns.barplot(data=diamonds, x='cut', y='price', order=cut_order,
            hue='cut', palette='Purples', estimator='median',
            errorbar=('ci', 95), legend=False, ax=axes[0, 2])
axes[0, 2].set_title('③ Median Price by Cut
⚠️ COUNTERINTUITIVE: Fair > Ideal!', fontweight='bold')
axes[0, 2].set_xlabel('Cut Quality'); axes[0, 2].set_ylabel('Median Price ($)')
axes[0, 2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── Panel 4: Carat distribution by cut (violin) ──
sns.violinplot(data=diamonds, x='cut', y='carat', order=cut_order,
               hue='cut', palette='Purples_r', inner='box',
               scale='width', legend=False, ax=axes[1, 0])
axes[1, 0].set_title('④ Carat by Cut (violin)
Fair cut = larger stones = higher price!', fontweight='bold')
axes[1, 0].set_xlabel('Cut Quality'); axes[1, 0].set_ylabel('Carat')

# ── Panel 5: Price per carat by cut (boxplot) ──
sns.boxplot(data=diamonds, x='cut', y='price_per_carat', order=cut_order,
            hue='cut', palette='Purples', legend=False,
            flierprops=dict(marker='.', alpha=0.2, markersize=3),
            ax=axes[1, 1])
axes[1, 1].set_title('⑤ Price/Carat by Cut
Ideal cut = HIGHEST price per carat ✅', fontweight='bold')
axes[1, 1].set_xlabel('Cut Quality'); axes[1, 1].set_ylabel('Price per Carat ($)')
axes[1, 1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── Panel 6: Carat vs price — hexbin (NB3, large dataset) ──
sns.jointplot(data=diamonds, x='carat', y='price', kind='hex',
              gridsize=28, cmap='YlOrRd',
              marginal_kws={'color': '#6A1B9A', 'fill': True, 'alpha': 0.6})
plt.suptitle('⑥ Carat vs Price — Hexbin
(Best for 53,940 points)', y=1.02,
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

fig.suptitle('💎 Case Study 3 — Diamond Distributions & Price Patterns',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig('outputs/case_studies/cs3_distributions.png', dpi=140, bbox_inches='tight')
plt.show()

print("📊 COUNTERINTUITIVE INSIGHT:")
print("   Fair cut has HIGHER median price than Ideal cut.")
print("   Reason: Fair-cut diamonds tend to be LARGER (more carats).")
print("   → Always analyse price PER CARAT, not raw price, when comparing quality grades.")

In [ ]:
# 🔴 Advanced | CS3 Step 3: Correlation + PairGrid (NB4 + NB5)

diamond_sample = diamonds.sample(n=2000, random_state=42)
num_cols_d = ['carat', 'depth', 'table', 'price', 'price_per_carat']
corr_d     = diamonds[num_cols_d].corr()
mask_d     = np.triu(np.ones_like(corr_d, dtype=bool))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Panel 1: Masked correlation heatmap ──
sns.heatmap(corr_d, mask=mask_d, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.8,
            cbar_kws={'shrink': 0.75}, ax=axes[0])
axes[0].set_title('① Diamond Feature Correlations
(carat vs price = r=0.92 — very strong)',
                  fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right', fontsize=9)

# ── Panel 2: Feature ranking with price ──
corr_price = (corr_d['price'].drop('price').abs().sort_values(ascending=True))
bar_cols_d = pd.Series(corr_price.values).map(
    lambda v: '#E53935' if v > 0.7 else '#FB8C00' if v > 0.3 else '#FDD835'
).tolist()
b2d = axes[1].barh(corr_price.index, corr_price.values,
                    color=bar_cols_d, edgecolor='white')
axes[1].bar_label(b2d, fmt='%.3f', fontsize=11, fontweight='bold', padding=4)
axes[1].set_title('② Feature Correlation with Price
carat dominates all others',
                  fontweight='bold')
axes[1].set_xlabel('|Pearson r| with price')
axes[1].axvline(0.7, color='#E53935', lw=1.2, linestyle='--', alpha=0.7)

fig.suptitle('💎 Case Study 3 — Diamond Feature Correlations',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig('outputs/case_studies/cs3_correlation.png', dpi=140, bbox_inches='tight')
plt.show()

# ── PairGrid — 3 features (NB5) ──
g = sns.PairGrid(diamond_sample[['carat', 'depth', 'price_per_carat']],
                 height=2.8)
g.map_diag(sns.histplot, kde=True, color='#6A1B9A', alpha=0.6, bins=20)
g.map_upper(sns.scatterplot, alpha=0.20, s=12, color='#6A1B9A')
g.map_lower(sns.kdeplot, fill=True, alpha=0.30, color='#6A1B9A', levels=5)
g.figure.suptitle('💎 Diamond PairGrid — Hist|Scatter|KDE
(carat vs price_per_carat shows expected positive curve)',
                  fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"📊 carat ↔ price correlation: r = {corr_d.loc['carat','price']:.3f} — extremely strong")
print(f"   depth ↔ price correlation: r = {corr_d.loc['depth','price']:.3f} — nearly zero")
print("   → carat is the dominant price driver; depth barely matters")

In [ ]:
# 🔴 Advanced | CS3 Step 4: lmplot + regression comparison (NB3)
# Do cut grades change how carat predicts price?

g = sns.lmplot(
    data=diamond_sample[diamond_sample['cut'].isin(['Fair', 'Good', 'Ideal'])],
    x='carat', y='price',
    hue='cut',
    col='cut',
    col_order=['Fair', 'Good', 'Ideal'],
    palette={'Fair': '#E53935', 'Good': '#FB8C00', 'Ideal': '#2E7D32'},
    scatter_kws={'alpha': 0.35, 's': 20},
    line_kws={'linewidth': 2.5},
    ci=95,
    height=4, aspect=1.1
)
g.set_axis_labels('Carat', 'Price ($)')
g.set_titles(col_template='Cut: {col_name}')
g.figure.suptitle(
    '💎 lmplot: Carat → Price by Cut Quality
'
    '(Similar slopes — cut grade modifies intercept, not the carat-price slope)',
    fontsize=12, fontweight='bold', y=1.04
)
plt.tight_layout()
g.figure.savefig('outputs/case_studies/cs3_lmplot.png', dpi=140, bbox_inches='tight')
plt.show()

print("📊 PRICING STRATEGY INSIGHT:")
print("   All cut grades show SAME slope (carat effect is consistent).")
print("   Ideal cut commands a HIGHER intercept (base price premium).")
print("   → Pricing formula: price ≈ 7,756×carat + cut_premium")
print("   → Recommend: market 'price-per-carat' advantage of Ideal vs Fair to customers.")

---

<div style="background: linear-gradient(90deg, #b71c1c, #c62828); padding: 20px 30px; border-radius: 12px; margin: 20px 0;">
  <h2 style="color: white; margin: 0; font-size: 1.8em;">✈️ Case Study 4 — Aviation Growth Analysis</h2>
  <p style="color: #ffcdd2; margin: 8px 0 0 0; font-size: 1.05em;">Dataset: <code>flights</code> | Goal: Understand seasonal patterns and long-term growth</p>
</div>

### 🏢 Business Context
An airline's strategy team wants:
1. What is the long-term passenger volume trend?
2. Are there strong seasonal patterns?
3. Which months are consistently peak vs off-peak?
4. Is growth accelerating or decelerating over time?

In [ ]:
# 🟡 Intermediate | CS4: Full time-series analysis (NB2, NB3)
flights = sns.load_dataset('flights')
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

# Engineer aggregations
yearly = flights.groupby('year')['passengers'].sum().reset_index()
yearly.columns = ['year', 'total']
yearly['yoy_pct'] = (yearly['total'].pct_change() * 100).round(2)
monthly_avg = flights.groupby('month')['passengers'].mean().reindex(month_order).reset_index()
monthly_avg.columns = ['month', 'avg_passengers']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── Panel 1: Yearly trend lineplot ──
sns.lineplot(data=yearly, x='year', y='total',
             color='#C62828', linewidth=2.8,
             marker='o', markersize=8,
             markerfacecolor='white', markeredgecolor='#C62828',
             markeredgewidth=2, ax=axes[0, 0])
axes[0, 0].fill_between(yearly['year'], yearly['total'], alpha=0.10, color='#C62828')
axes[0, 0].set_title('① Annual Passenger Volume 1949–1960
(Consistent exponential growth)',
                      fontweight='bold')
axes[0, 0].set_xlabel('Year'); axes[0, 0].set_ylabel('Total Passengers')
axes[0, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[0, 0].set_xticks(yearly['year'])

# Annotate start and end
axes[0, 0].annotate(f"{yearly['total'].iloc[0]:,}", xy=(1949, yearly['total'].iloc[0]),
                    xytext=(1950, yearly['total'].iloc[0] + 200),
                    fontsize=9, color='#C62828', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.5))
axes[0, 0].annotate(f"{yearly['total'].iloc[-1]:,}", xy=(1960, yearly['total'].iloc[-1]),
                    xytext=(1958, yearly['total'].iloc[-1] + 200),
                    fontsize=9, color='#C62828', fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.5))

# ── Panel 2: Monthly seasonality ──
sns.lineplot(data=monthly_avg, x='month', y='avg_passengers',
             color='#C62828', linewidth=2.5,
             marker='D', markersize=7, ax=axes[0, 1])
axes[0, 1].fill_between(monthly_avg.index, monthly_avg['avg_passengers'], alpha=0.12, color='#C62828')
axes[0, 1].axhspan(monthly_avg['avg_passengers'].quantile(0.75),
                    monthly_avg['avg_passengers'].max() * 1.05,
                    alpha=0.08, color='#E53935', label='Peak season')
axes[0, 1].set_title('② Monthly Seasonality Pattern
(Summer = peak | Winter = trough)',
                      fontweight='bold')
axes[0, 1].set_xlabel('Month'); axes[0, 1].set_ylabel('Avg Passengers')
axes[0, 1].set_xticks(range(12)); axes[0, 1].set_xticklabels(month_order, rotation=30)
axes[0, 1].legend(fontsize=9)

# ── Panel 3: twinx — volume + YoY growth ──
ax_left  = axes[1, 0]
ax_right = axes[1, 0].twinx()

bars3 = ax_left.bar(yearly['year'], yearly['total'],
                     color='#FFCDD2', alpha=0.85, width=0.6, label='Total Passengers')
ax_right.plot(yearly.dropna()['year'], yearly.dropna()['yoy_pct'],
              color='#B71C1C', linewidth=2.5, marker='s', markersize=7,
              markerfacecolor='white', markeredgecolor='#B71C1C', markeredgewidth=2,
              label='YoY Growth %')
ax_right.axhline(0, color='#B71C1C', lw=1, linestyle='--', alpha=0.3)
ax_left.set_ylabel('Total Passengers', color='#E57373', fontsize=10)
ax_right.set_ylabel('YoY Growth (%)',  color='#B71C1C', fontsize=10)
ax_left.tick_params(axis='y', labelcolor='#E57373')
ax_right.tick_params(axis='y', labelcolor='#B71C1C')
h1, l1 = ax_left.get_legend_handles_labels()
h2, l2 = ax_right.get_legend_handles_labels()
ax_left.legend(h1 + h2, l1 + l2, fontsize=9, loc='upper left')
axes[1, 0].set_title('③ Volume (bars) + YoY Growth % (line)
(twinx: dual metric analysis)',
                      fontweight='bold')
ax_left.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# ── Panel 4: Heatmap — month × year passenger matrix ──
pivot_flights = flights.pivot(index='month', columns='year', values='passengers')
pivot_flights = pivot_flights.reindex(month_order)
sns.heatmap(pivot_flights, cmap='YlOrRd', annot=True, fmt='d',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Passengers', 'shrink': 0.75},
            ax=axes[1, 1])
axes[1, 1].set_title('④ Passenger Heatmap: Month × Year
(Darkening over time + clear summer peaks)',
                      fontweight='bold')
axes[1, 1].set_xlabel('Year'); axes[1, 1].set_ylabel('Month')

fig.suptitle('✈️ Case Study 4 — Aviation: Complete Trend & Seasonality Analysis',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig('outputs/case_studies/cs4_aviation.png', dpi=140, bbox_inches='tight')
plt.show()

avg_growth = yearly['yoy_pct'].mean()
peak_month = monthly_avg.loc[monthly_avg['avg_passengers'].idxmax(), 'month']
slow_month = monthly_avg.loc[monthly_avg['avg_passengers'].idxmin(), 'month']
print(f"📊 KEY METRICS:")
print(f"   Avg YoY growth:   {avg_growth:.1f}%")
print(f"   Peak month:       {peak_month}")
print(f"   Slowest month:    {slow_month}")
print(f"   3.8× growth 1949→1960")
print()
print("📊 BUSINESS RECOMMENDATIONS (Aviation):")
print("   1. Maximize pricing and capacity in Jul–Aug (summer peak)")
print("   2. Offer aggressive discounts in Jan–Feb to fill seats (trough months)")
print("   3. Growth rate trending STABLE ~10% YoY — plan fleet expansion accordingly")

In [ ]:
# 🔴 Advanced | CS4: FacetGrid — seasonal patterns by year groups (NB5)
# Group years into two halves and compare seasonal patterns.

flights['decade_half'] = pd.cut(
    flights['year'],
    bins=[1948, 1954, 1960],
    labels=['1949–1954 (Early era)', '1955–1960 (Growth era)']
)

g = sns.FacetGrid(
    flights,
    col='decade_half',
    height=4.5, aspect=1.4,
    sharey=False
)
g.map_dataframe(
    sns.lineplot,
    x='month', y='passengers',
    hue='year', palette='Reds',
    estimator=None,           # Plot each year as a separate line
    linewidth=1.5, alpha=0.75,
    legend=False
)
# Add mean line per panel explicitly
ax_early  = g.axes_dict['1949–1954 (Early era)']
ax_growth = g.axes_dict['1955–1960 (Growth era)']

early_mean  = flights[flights['decade_half']=='1949–1954 (Early era)'].groupby('month')['passengers'].mean().reindex(month_order)
growth_mean = flights[flights['decade_half']=='1955–1960 (Growth era)'].groupby('month')['passengers'].mean().reindex(month_order)

ax_early.plot(range(12),  early_mean.values,  color='black', lw=2.5,
              linestyle='--', label='Period mean')
ax_growth.plot(range(12), growth_mean.values, color='black', lw=2.5,
               linestyle='--', label='Period mean')

ax_early.legend(fontsize=9); ax_growth.legend(fontsize=9)
ax_early.set_xticks(range(12));  ax_early.set_xticklabels(month_order, rotation=35, fontsize=8)
ax_growth.set_xticks(range(12)); ax_growth.set_xticklabels(month_order, rotation=35, fontsize=8)
ax_early.set_ylabel('Passengers');  ax_growth.set_ylabel('')

g.set_titles(col_template='{col_name}')
g.figure.suptitle(
    '✈️ Seasonal Pattern Comparison: Early Era vs Growth Era
'
    '(Same seasonal shape — summer peak consistent; absolute volume doubled)',
    fontsize=12, fontweight='bold', y=1.04
)
plt.tight_layout()
g.figure.savefig('outputs/case_studies/cs4_facetgrid_seasons.png', dpi=140, bbox_inches='tight')
plt.show()

print("📊 INSIGHT: The seasonal SHAPE is consistent across both eras.")
print("   The peak-to-trough ratio stays similar — seasonal demand is structural.")
print("   Only the baseline volume increases year over year.")

---

## 🗺️ Master Concept Map — What Was Applied Where

| Concept | NB Source | CS1 Tips | CS2 Titanic | CS3 Diamonds | CS4 Flights |
|---------|-----------|----------|-------------|--------------|-------------|
| `sns.set_theme()` | NB1 | ✅ | ✅ | ✅ | ✅ |
| `histplot` | NB2 | ✅ | ✅ | ✅ | — |
| `kdeplot` | NB2 | ✅ | ✅ | — | — |
| `countplot` | NB2 | ✅ | ✅ | ✅ | — |
| `barplot` | NB2 | ✅ | ✅ | ✅ | — |
| `boxplot` | NB2 | ✅ | — | ✅ | — |
| `violinplot` | NB2 | ✅ | ✅ | ✅ | — |
| `scatterplot` (annotated) | NB3 | ✅ | — | — | — |
| `lineplot` | NB3 | — | — | — | ✅ |
| `regplot` / `lmplot` | NB3 | ✅ | — | ✅ | — |
| Masked `heatmap` | NB4 | ✅ | ✅ | ✅ | — |
| Missing value heatmap | NB4 | — | ✅ | — | — |
| Feature correlation ranking | NB4 | ✅ | ✅ | ✅ | — |
| `pairplot` / `PairGrid` | NB4/NB5 | — | — | ✅ | — |
| `jointplot` (hex) | NB4 | — | — | ✅ | — |
| 2D survival rate heatmap | NB4 | — | ✅ | — | — |
| `FacetGrid.map_dataframe` | NB5 | — | ✅ | — | ✅ |
| `ax.twinx()` | NB5 | ✅ | — | — | ✅ |
| `ax.annotate()` | NB5 | ✅ | — | — | ✅ |
| Reusable functions | NB5 | ✅ | ✅ | ✅ | ✅ |
| Palette selection | NB1 | ✅ | ✅ | ✅ | ✅ |

> 🎯 **Total unique concepts applied: 22 across 4 case studies**

---

## 🎤 Case Study Interview Questions

These are questions interviewers ask when you present a project on these datasets.

---

**Q (Titanic): "What feature is the strongest predictor of survival?"**
> *"Passenger class (pclass) has the strongest numeric correlation (r = −0.34) with survival. Sex is categorical but even more powerful — the class × sex survival matrix shows 1st class female at ~97% vs 3rd class male at ~17%. In an ML model, I'd one-hot encode sex and class, and they'd dominate feature importance."*

---

**Q (Diamonds): "Why is Fair-cut more expensive than Ideal-cut on average?"**
> *"This is a classic example of Simpson's Paradox. Fair-cut diamonds tend to be physically larger (more carats), and carat is by far the strongest price driver (r = 0.92). When you control for carat — comparing price per carat — Ideal-cut actually commands a higher price. The lesson: always normalise before comparing across groups."*

---

**Q (Restaurant): "If I had to build a simple tip prediction model, what would you use?"**
> *"Start with `total_bill` alone — it explains 45.7% of tip variance (R² = 0.457), which is good for a single feature. The linear relationship is appropriate (LOWESS confirms the pattern isn't curved). The model would be: tip ≈ 0.105 × total_bill + 0.92. Adding party size as a second feature would improve it further."*

---

**Q (Aviation): "Is the growth sustainable?"**
> *"The average YoY growth rate was ~10.3%, which was driven by post-war economic expansion and commercial aviation democratisation. The heatmap shows the seasonal pattern (summer peak) is extremely consistent — it's structural. Growth sustainability would depend on capacity (aircraft), regulatory environment, and oil prices — factors not in this dataset, but the trend data gives a strong baseline."*

---

## 📝 Final Practice — Integrated Questions

**Q1. Cross-dataset.** Using `tips` and `penguins` as analogies, explain why you should never use `stat='count'` in `histplot` when comparing groups of different sizes. Draw from both Case Study 1 and your NB2 knowledge.

**Q2. End-to-end.** Choose any one case study. Write the complete EDA workflow as bullet points — from data loading to business recommendation — naming every Seaborn function you'd use and why.

**Q3. Debugging.** A colleague says: *"I built a correlation heatmap and it shows tip_pct is perfectly correlated with tip. Great — I'll use it as a feature to predict tip amount!"* What is wrong with this reasoning? (Hint: NB4 data leakage discussion.)

**Q4. Design.** A manager asks for a single chart that shows: (a) how many Titanic passengers were in each class, (b) what fraction of each class survived. Which Seaborn plots would you combine and how?

**Q5. Production.** Write the function signature (parameters and docstring only — no body) for a reusable `run_case_study_eda(df, target_col, cat_cols, dataset_name)` function that automates the complete EDA pipeline shown across these 4 case studies.

---

<div style="background: linear-gradient(135deg, #0a0a0a 0%, #1a1a2e 50%, #0f3460 100%); padding: 35px; border-radius: 14px; text-align: center; margin-top: 40px; border: 1px solid #e94560;">
  <h2 style="color: #e94560; font-size: 2em;">🏆 Notebook 6 — Master EDA Complete!</h2>
  <p style="color: #f5f5f5; font-size: 1.1em;">You've applied every Seaborn concept across 4 real-world business domains.</p>
  <p style="color: #a8dadc; font-size: 1em; margin-top: 8px;">Restaurant · Survival Analysis · Luxury Retail · Time Series</p>
  <hr style="border: 1px solid #e94560; margin: 18px auto; width: 55%;">
  <p style="color: #888;">📚 Final Notebook: <strong style='color:#f5f5f5'>Notebook 7</strong> — Interview Revision Guide</p>
  <p style="color: #666; font-size: 0.9em;">Quick-fire Q&A · Trap Questions · API cheatsheet · Decision flowchart · Bug Gallery</p>
</div>